In [1]:
import pandas as pd
import numpy as np

# --- MOCK MODULES (Representing your separate files) ---

def run_proxi_link_step(true_dist):
    # Simulate RSSI and return "State"
    rssi = -65 - (10 * 2.4 * np.log10(true_dist)) + np.random.normal(0, 2)
    state = "INTERACTING" if rssi > -68 else "SAFE"
    return state

def run_aegis_drive_step(current_v, dist_to_wall, human_detected):
    # If human is detected by ProxiLink, force braking regardless of wall distance
    decel = 3.0
    if human_detected == "INTERACTING" or dist_to_wall < 1.0:
        new_v = max(0, current_v - (decel * 0.1))
        action = "BRAKING"
    else:
        new_v = min(1.2, current_v + (0.5 * 0.1))
        action = "CRUISING"
    return new_v, action

def run_forge_sight_audit(v_history, action_history):
    # Predictive Maintenance: High frequency braking increases failure probability
    braking_events = action_history.count("BRAKING")
    failure_risk = min(100, braking_events * 2.5) # Simplified risk logic
    return failure_risk

# --- THE UNIFIED HUB EXECUTION ---

v = 1.2
dist = 5.0
history = []

print("🚀 INDUSTRIAL HUB STARTING...")
print(f"{'Time':<6} | {'ProxiLink':<12} | {'Aegis-V':<8} | {'ForgeRisk':<10}")
print("-" * 50)

for t in np.arange(0, 5, 0.5):
    # 1. ProxiLink Check (Let's simulate a person walking by at t=2.0)
    sim_human_dist = 0.5 if 2.0 <= t <= 3.5 else 5.0
    human_state = run_proxi_link_step(sim_human_dist)
    
    # 2. Aegis-Drive Decision
    v, vehicle_action = run_aegis_drive_step(v, dist, human_state)
    dist -= v * 0.5
    
    # 3. ForgeSight Monitoring
    actions_so_far = [h[3] for h in history] + [vehicle_action]
    risk = run_forge_sight_audit(None, actions_so_far)
    
    history.append([t, human_state, v, vehicle_action, risk])
    
    print(f"{t:<6.1f} | {human_state:<12} | {v:<8.2f} | {risk:>8.1f}%")

print("-" * 50)
print("✅ OPERATION COMPLETE: System Logs exported for ForgeSight Training.")

🚀 INDUSTRIAL HUB STARTING...
Time   | ProxiLink    | Aegis-V  | ForgeRisk 
--------------------------------------------------
0.0    | SAFE         | 1.20     |      0.0%
0.5    | SAFE         | 1.20     |      0.0%
1.0    | SAFE         | 1.20     |      0.0%
1.5    | SAFE         | 1.20     |      0.0%
2.0    | INTERACTING  | 0.90     |      2.5%
2.5    | INTERACTING  | 0.60     |      5.0%
3.0    | INTERACTING  | 0.30     |      7.5%
3.5    | INTERACTING  | 0.00     |     10.0%
4.0    | SAFE         | 0.05     |     10.0%
4.5    | SAFE         | 0.10     |     10.0%
--------------------------------------------------
✅ OPERATION COMPLETE: System Logs exported for ForgeSight Training.
